# CogMem Cognitive Patches — Minimal Experiment

**Goal:** Verify the cognitive patches architecture works.

**Plan:**
1. Load base model (4-bit, ~2GB VRAM)
2. Process first 100 tasks with N=4 candidates per task
3. Create patches from pass/fail contrasts (~20 patches expected)
4. Evaluate patches on remaining 1040 UNSEEN tasks
5. Compare: patched eval > cold eval = architecture works

**Hardware:** A4000 16GB. Model loaded in 4-bit via transformers (not Ollama).
Generation happens through model.generate() directly.


In [1]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers -q
print("Deps installed")


NVIDIA RTX A4000, 16376 MiB, 16101 MiB
Deps installed


In [2]:
!pip install "transformers==4.43.4" "sentence-transformers==2.7.0" "huggingface-hub==0.25.0" "accelerate==0.33.0" "peft==0.13.2" "bitsandbytes==0.43.3" -q


In [28]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    README.md
	deleted:    quick_start_pytorch.ipynb
	deleted:    quick_start_pytorch_images/example_instance_type.png

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.Trash-0/
	.abc123/
	.ipynb_checkpoints/
	.jpg/
	.memos/
	.pdf/
	.txt
	.xyz/
	CogMem/
	MemRL/
	Modelfile.cogmem_bigcode
	PREFIX__data.json
	PREFIX__empty.json
	PREFIX__temp.json
	acrotray_exe_archive.tar
	all_custom.zip
	all_custom/
	all_custom_14443/
	archive.tar
	backup/
	bigcode_episodes.jsonl
	bigcodebench_hard_tasks.jsonl
	bigcodebench_tasks.jsonl
	blobs_distance_plot.png
	cogmem_merged/
	cogmem_patches/
	cumulative_sum_chart.png
	custom_directory.zip
	custom_directory/
	custom_filename.pkl
	data.json
	data.yaml
	df_contents.txt
	dice_distribution.

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [37]:
# Cell 2: Clone CogMem + load tasks
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || \
    (cd /notebooks/CogMem && git pull)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import json
from pathlib import Path
from datasets import load_dataset

# Load BigCodeBench full (1140 tasks)
TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(TASKS_PATH, "w") as f:
        for t in tasks:
            f.write(json.dumps(t) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

print("Tasks:", len(tasks))
# Split: first 100 for patch creation, rest for evaluation
TRAIN_TASKS = tasks[:100]
EVAL_TASKS = tasks[100:]
print("Train (create patches):", len(TRAIN_TASKS))
print("Eval (test patches):", len(EVAL_TASKS))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


remote: Enumerating objects: 10, done.
remote: Total 10 (delta 0), reused 0 (delta 0), pack-reused 10 (from 1)
Unpacking objects: 100% (10/10), 5.25 KiB | 79.00 KiB/s, done.
From https://github.com/tungooxx/CogMem
   ae201ba..e35927f  master     -> origin/master
Updating ae201ba..e35927f
Fast-forward
 cogmem/patches/create.py | 8 ++++++--
 1 file changed, 6 insertions(+), 2 deletions(-)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Tasks: 1140
Train (create patches): 100
Eval (test patches): 1040


In [38]:
# Cell 3: Load base model (4-bit) + embedder
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading embedder...")
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Model loaded. Free VRAM: {free:.1f} GB")
print("Ready for patch creation.")


Loading model (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading embedder...
Model loaded. Free VRAM: 6.3 GB
Ready for patch creation.


In [ ]:
# Cell 4: Record episodes from first 100 tasks
import time
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.memory_bank import ClusterMemoryBank
from cogmem.patches.wake import generate_with_model, find_best_contrast_pair

N_CANDIDATES = 8
MEMORY_DIR = "/notebooks/cogmem_cluster_memories"
memory_bank = ClusterMemoryBank(MEMORY_DIR)
memory_bank.load()

from peft import prepare_model_for_kbit_training
base_model = prepare_model_for_kbit_training(base_model)
print('Base model prepared for training')

start_time = time.time()
episodes_before = len(memory_bank.episodes)
total_passed = 0

for i, task in enumerate(TRAIN_TASKS):
    task_id = task["task_id"]
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    candidates = []
    for _ in range(N_CANDIDATES):
        try:
            response = generate_with_model(base_model, tokenizer, messages, temperature=0.8)
            code = extract_code(response)
            if code and len(code.strip()) > 20:
                result = evaluate_solution(task, code, timeout=30, mode="subprocess")
                candidates.append({"code": code, "passed": result["passed"]})
        except Exception as e:
            if i < 3:
                print('  Gen error:', type(e).__name__, str(e)[:80])

    passes = [c for c in candidates if c["passed"]]
    fails = [c for c in candidates if not c["passed"]]

    if passes:
        total_passed += 1

    if passes and fails:
        best_pair, best_sim = find_best_contrast_pair(passes, fails)
        if best_pair:
            memory_bank.record_episode(
                task_id=task_id,
                prompt=prompt,
                task_embedding=task_embedding,
                failed_code=best_pair["fail"]["code"],
                passed_code=best_pair["pass"]["code"],
                pass_fail_similarity=best_sim,
            )

    if (i + 1) % 10 == 0 or i < 5:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
        print("[{}/{}] {}: {}P/{}F | episodes={} | pass_rate={}/{} | {:.0f}/hr".format(
            i + 1, len(TRAIN_TASKS), task_id,
            len(passes), len(fails), len(memory_bank.episodes),
            total_passed, i + 1, rate))

memory_bank.save()
elapsed = (time.time() - start_time) / 60
print()
print("=" * 50)
print("EPISODE RECORDING COMPLETE")
print("Tasks processed:", len(TRAIN_TASKS))
print("Tasks with passes:", total_passed)
print("New episodes recorded:", len(memory_bank.episodes) - episodes_before)
print("Time:", round(elapsed, 1), "min")
print("Memory bank stats:", memory_bank.stats())


In [ ]:
# Cell 4b: Build cluster memories and inspect distilled artifacts
import torch
from cogmem.patches.compose import PatchedModel

build_stats = memory_bank.build_memories(base_model, tokenizer)
retrievable = [m for m in memory_bank.memories if m.retrievable]
print('Memory bank stats:', build_stats)
print('Retrievable memories:', len(retrievable))

print('\n=== CLUSTER MEMORY SUMMARY ===')
for memory in memory_bank.memories[:10]:
    print('Memory:', memory.memory_id)
    print('  family:', memory.family_label, 'support:', memory.support_count, 'q:', round(memory.q_value, 3))
    print('  explained variance:', [round(v, 3) for v in memory.explained_variance[:3]])
    print('  held-out gain:', round(memory.held_out_steering_gain, 4),
          'penalty:', round(memory.negative_steering_penalty, 4),
          'transfer:', round(memory.transfer_rate, 3),
          'distill:', round(memory.distillation_success, 3))
    print('  patches:', memory.distilled_patch_ids)

if not retrievable:
    print('\nNo retrievable memories yet. Add more episodes or inspect family clustering.')
else:
    print('\n[1] Distilled artifact magnitudes:')
    for memory in retrievable[:3]:
        active = memory_bank.load_patches_for_memories([memory])
        if not active:
            print('  {}: no artifact patch loaded'.format(memory.memory_id))
            continue
        patch = active[0]
        norms = []
        for _, w in patch.lora_weights.items():
            nA = torch.norm(w['A']).item()
            nB = torch.norm(w['B']).item()
            norms.append(nA + nB)
        avg = sum(norms) / len(norms) if norms else 0.0
        first_key = list(patch.lora_weights.keys())[0]
        w = patch.lora_weights[first_key]
        print('  {} -> {}: |A|={:.4f} |B|={:.4f} avg_norm={:.4f}'.format(
            memory.memory_id[:36], patch.patch_id[:36],
            torch.norm(w['A']).item(), torch.norm(w['B']).item(), avg))
        patch.unload_weights()

    print('\n[2] Output difference (first 3 eval tasks):')
    for task in EVAL_TASKS[:3]:
        prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
        emb = embedder.encode(prompt).tolist()
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]

        torch.manual_seed(42)
        out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

        active_memories = memory_bank.get_active_memories(emb, prompt, top_k=5)
        active = memory_bank.load_patches_for_memories(active_memories)

        torch.manual_seed(42)
        try:
            with PatchedModel(base_model, active, scaling_factor=0.25):
                out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)
        finally:
            for patch in active:
                patch.unload_weights()

        if out_cold == out_patched:
            print('  {}: IDENTICAL'.format(task['task_id']))
        else:
            cold_tokens = out_cold.split()
            patched_tokens = out_patched.split()
            diff = sum(1 for a, b in zip(cold_tokens, patched_tokens) if a != b)
            total = max(len(cold_tokens), len(patched_tokens), 1)
            print('  {}: {:.0f}% tokens different | memories={}'.format(
                task['task_id'], diff / total * 100, [m.memory_id for m in active_memories]))


In [ ]:
from cogmem.patches.memory_bank import ClusterMemoryBank
memory_bank = ClusterMemoryBank("/notebooks/cogmem_cluster_memories")
memory_bank.load()
print(f"Episodes: {len(memory_bank.episodes)}")
print(f"Memories: {len(memory_bank.memories)}")
print(f"Artifact patches: {len(memory_bank.artifact_bank.patches)}")


In [ ]:
# Cell 4c: Check composition works with retrieved cluster memories
import torch
from cogmem.patches.compose import PatchedModel

task = EVAL_TASKS[0]
prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
emb = embedder.encode(prompt).tolist()
messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]

active_memories = memory_bank.get_active_memories(emb, prompt, top_k=5)
active = memory_bank.load_patches_for_memories(active_memories)
print('Selected memories:', [m.memory_id for m in active_memories])
print('Artifact patches:', [p.patch_id for p in active])

print()
print('=== Greedy (temp=0) ===')
torch.manual_seed(42)
out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

torch.manual_seed(42)
with PatchedModel(base_model, active, scaling_factor=0.25):
    out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)

print('Cold first 100:', out_cold[:100])
print('Patched first 100:', out_patched[:100])
print('IDENTICAL:', out_cold == out_patched)

print()
print('=== Hook Count ===')
pm = PatchedModel(base_model, active, scaling_factor=0.25)
pm.__enter__()
print('Hooks registered:', len(pm._hooks))
pm.__exit__(None, None, None)
print('Hooks removed:', len(pm._hooks) == 0)

print()
print('=== Low-temp (0.01) ===')
torch.manual_seed(42)
out_cold_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

torch.manual_seed(42)
with PatchedModel(base_model, active, scaling_factor=0.25):
    out_patched_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

print('IDENTICAL:', out_cold_lt == out_patched_lt)

for patch in active:
    patch.unload_weights()


In [ ]:
# Cell 5: Evaluate cluster memories on unseen tasks
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.compose import PatchedModel
from cogmem.patches.wake import generate_with_model

EVAL_SIZE = 200
EVAL_TOP_K = 5
EVAL_SCALE = 0.25
eval_subset = EVAL_TASKS[:EVAL_SIZE]
print("Evaluating on", EVAL_SIZE, "unseen tasks")
print("Episodes:", len(memory_bank.episodes))
print("Memories:", len(memory_bank.memories))
print("Artifact patches:", len(memory_bank.artifact_bank.patches))

print()
print("--- COLD EVAL (base model, no memories) ---")
cold_passed = 0
for i, task in enumerate(eval_subset):
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")
        if result["passed"]:
            cold_passed += 1
    except Exception:
        pass

    if (i + 1) % 50 == 0:
        print("  [{}/{}] cold: {}/{} ({:.1%})".format(
            i + 1, EVAL_SIZE, cold_passed, i + 1, cold_passed / (i + 1)))

cold_rate = cold_passed / max(EVAL_SIZE, 1)
print("Cold result:", cold_passed, "/", EVAL_SIZE, "({:.1%})".format(cold_rate))

print()
print("--- MEMORY EVAL (base model + retrieved distilled patches) ---")
patched_passed = 0
for i, task in enumerate(eval_subset):
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()
    active_memories = memory_bank.get_active_memories(task_embedding, prompt, top_k=EVAL_TOP_K)
    active_patches = memory_bank.load_patches_for_memories(active_memories)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    try:
        with PatchedModel(base_model, active_patches, scaling_factor=EVAL_SCALE):
            response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")
        if result["passed"]:
            patched_passed += 1
    except Exception:
        pass
    finally:
        for patch in active_patches:
            patch.unload_weights()

    if (i + 1) % 50 == 0:
        print("  [{}/{}] memory: {}/{} ({:.1%})".format(
            i + 1, EVAL_SIZE, patched_passed, i + 1, patched_passed / (i + 1)))

patched_rate = patched_passed / max(EVAL_SIZE, 1)
print("Memory result:", patched_passed, "/", EVAL_SIZE, "({:.1%})".format(patched_rate))


In [ ]:
# Cell 6: Results comparison
print("=" * 50)
print("EPISODE-FIRST CLUSTER MEMORY RESULTS")
print("=" * 50)
print()
print("Episodes recorded:", len(memory_bank.episodes))
print("Cluster memories built:", len(memory_bank.memories))
print("Artifact patches available:", len(memory_bank.artifact_bank.patches))
print("Evaluated on", EVAL_SIZE, "UNSEEN tasks")
print()
print("{:<20} {:>8} {:>8} {:>10}".format("Model", "Passed", "Total", "Rate"))
print("-" * 48)
print("{:<20} {:>8} {:>8} {:>9.1%}".format("Cold", cold_passed, EVAL_SIZE, cold_rate))
print("{:<20} {:>8} {:>8} {:>9.1%}".format("Cluster memory", patched_passed, EVAL_SIZE, patched_rate))
print()
diff = patched_rate - cold_rate
if diff > 0.01:
    print("Cluster memories IMPROVED by {:.1%}.".format(diff))
elif diff > -0.01:
    print("No significant difference yet.")
else:
    print("Cluster memories HURT by {:.1%}.".format(abs(diff)))

print()
print("Memory bank:")
for key, value in memory_bank.stats().items():
    print("  {}: {}".format(key, value))


In [ ]:
# Cell 7: Inspect cluster memories and retrieval concentration
from collections import Counter

for memory in memory_bank.memories[:5]:
    print("Memory:", memory.memory_id)
    print("  Family:", memory.family_label)
    print("  Support:", memory.support_count)
    print("  Q:", round(memory.q_value, 3), "reuse:", memory.reuse_count)
    print("  Gain:", round(memory.held_out_steering_gain, 4),
          "Penalty:", round(memory.negative_steering_penalty, 4),
          "Transfer:", round(memory.transfer_rate, 3))
    print("  Distilled patches:", memory.distilled_patch_ids)
    print()

retrieval_counts = Counter()
family_counts = Counter()
for task in EVAL_TASKS[:50]:
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()
    active_memories = memory_bank.get_active_memories(task_embedding, prompt, top_k=5)
    for memory in active_memories:
        retrieval_counts[memory.memory_id] += 1
        family_counts[memory.family_label] += 1

print("Top retrieved memories:")
for memory_id, hits in retrieval_counts.most_common(10):
    print("  {} -> {} hits".format(memory_id, hits))

print("\nRetrieved family concentration:")
for family, hits in family_counts.most_common():
    print("  {} -> {} hits".format(family, hits))
